# PTMs Augmentation Data Preparation
**Summary:** Prepares and splits data for PTM augmentation. Merges sequences with amino acid substitutions and prepares train/val/holdout splits using Iterative Stratification.

**Required Files:**
- `<PATH_TO_META_DATA_PARQUET>`: e.g., TUM third pool meta data
- `<PATH_TO_AA_SUBS_CSV>`: e.g., aa_subs.csv
- `<PATH_TO_NEW_META_DATA_DIR>`: Directory containing additional parquet metadata files

## Configuration
Define all required paths.

In [ ]:
META_DATA_PARQUET = '<PATH_TO_META_DATA_PARQUET>'
AA_SUBS_CSV = '<PATH_TO_AA_SUBS>'
NEW_META_DATA_DIR = '<PATH_TO_NEW_META_DATA>'
TRAIN_OUT = '<PATH_TO_TRAIN_OUT>'
VAL_OUT = '<PATH_TO_VAL_OUT>'
HO_OUT = '<PATH_TO_HO_OUT>'


In [ ]:
import pandas as pd
from glob import iglob
from tqdm import tqdm
import argparse
import random
import collections
import numpy
import numpy as np
import pandas as pd
from skmultilearn.model_selection import IterativeStratification
from spectrum_fundamentals.mod_string import internal_without_mods
from spectrum_fundamentals.constants import AA_ALPHABET



In [ ]:
def argshuffle_splits(idx, traininsplit):
    train_idx = list(numpy.random.permutation(idx[:traininsplit]))
    test_idx = list(numpy.random.permutation(idx[traininsplit:]))
    return train_idx, test_idx


def peptide_argsort(sequence_integer, seed=42):
    random.seed(seed)
    peptide_groups = collections.defaultdict(list)
    for index, row in enumerate(sequence_integer):
        peptide_groups[tuple(row)].append(index)

    # shuffle within peptides
    for indeces in peptide_groups.values():
        random.shuffle(indeces)

    # shuffle peptides
    peptides = list(peptide_groups.keys())
    random.shuffle(peptides)

    # join indeces
    indeces = []
    for peptide in peptides:
        indeces.extend(peptide_groups[peptide])
    return indeces

meta_data = pd.read_parquet(META_DATA_PARQUET)

meta_data

AA_ALPHABET['C']=2


## Iterative Stratification Split
Split the substitutions maintaining balanced representation.

In [ ]:
subs = pd.read_csv(AA_SUBS_CSV)
subs['from_aa_int'] = subs['from_aa'].apply(lambda x: AA_ALPHABET[x])
subs['to_aa_int'] = subs['to_aa'].apply(lambda x: AA_ALPHABET[x])
subs = subs[subs['from_aa_int']<subs['to_aa_int']]





def iterative_split(df, test_size, stratify_columns):
    """Custom iterative train test split which
    'maintains balanced representation with respect
    to order-th label combinations.'

    From https://madewithml.com/courses/mlops/splitting/#stratified-split
    """
    # One-hot encode the stratify columns and concatenate them
    one_hot_cols = [pd.get_dummies(df[col]) for col in stratify_columns]
    one_hot_cols = pd.concat(one_hot_cols, axis=1).to_numpy()
    stratifier = IterativeStratification(
        n_splits=2, order=len(stratify_columns), sample_distribution_per_fold=[test_size, 1-test_size])
    train_indices, test_indices = next(stratifier.split(df.to_numpy(), one_hot_cols))
    # Return the train and test set dataframes
    train, test = df.iloc[train_indices], df.iloc[test_indices]
    return train, test


In [ ]:
def get_mz_cat(mz):
    if mz<20:
        return 1
    elif mz<40:
        return 2
    elif mz<60:
        return 3
    elif mz<80:
        return 4
    elif mz<100:
        return 5
    elif mz>100:
        return 6

def get_d95_pred_cat(d95_pred):
    if d95_pred<0.3:
        return 1
    elif d95_pred<0.5:
        return 2
    elif d95_pred<0.7:
        return 3
    elif d95_pred<0.8:
        return 4
    elif d95_pred<0.9:
        return 5
    else:
        return 6


subs['mz_cat'] = subs['abs_delta_mono_mass'].apply(lambda x: get_mz_cat(x))
subs['d95_pred_cat'] = subs['SA_pred'].apply(lambda x: get_d95_pred_cat(x))




train, test = iterative_split(subs, 0.25, ['mz_cat', 'd95_pred_cat'])


val, ho = iterative_split(test, 0.5, ['mz_cat', 'd95_pred_cat'])


In [ ]:
subs = pd.read_csv(AA_SUBS_CSV)
subs['from_aa_int'] = subs['from_aa'].apply(lambda x: AA_ALPHABET[x])
subs['to_aa_int'] = subs['to_aa'].apply(lambda x: AA_ALPHABET[x])
subs = subs[subs['from_aa_int']>subs['to_aa_int']]



rows_to_add = []
for _,row in train.iterrows():
    current_row = subs[(subs['from_aa_int']==row['to_aa_int']) & (subs['to_aa_int']==row['from_aa_int'])]
    rows_to_add.append(current_row)
    
rows_to_add_test = []
for _,row in val.iterrows():
    current_row = subs[(subs['from_aa_int']==row['to_aa_int']) & (subs['to_aa_int']==row['from_aa_int'])]
    rows_to_add_test.append(current_row)
    
rows_to_add_ho = []
for _,row in ho.iterrows():
    current_row = subs[(subs['from_aa_int']==row['to_aa_int']) & (subs['to_aa_int']==row['from_aa_int'])]
    rows_to_add_ho.append(current_row)

reverse_train = pd.concat(rows_to_add)
reverse_val = pd.concat(rows_to_add_test)
reverse_ho = pd.concat(rows_to_add_ho)

train = pd.concat([train,reverse_train])
val = pd.concat([val,reverse_val])
ho = pd.concat([ho,reverse_ho])





train.to_csv(TRAIN_OUT,index=False)
val.to_csv(VAL_OUT,index=False)
ho.to_csv(HO_OUT,index=False)


## Sequence Augmentation
Perform random replacements to augment the modified sequences.

In [ ]:
def random_replace(s, replace_from, replace_to):
    modified_sequence = s
    parts = modified_sequence.split(replace_from)
    indices = random.sample(range(len(parts) - 1),1)

    replaced_s_parts = list()

    for i in range(len(parts)):
        replaced_s_parts.append(parts[i])
        if i < len(parts) - 1:
            if i in indices:
                replaced_s_parts.append(replace_to)
            else:
                replaced_s_parts.append(replace_from)
    return "".join(replaced_s_parts)

def split_given_size(a, size):
    return np.split(a, np.arange(size,len(a),size))

meta_data_files = [f for f in iglob(NEW_META_DATA_DIR + '/*')]

meta_data_files

meta_data = []

for f in tqdm(meta_data_files):
    df_meta = pd.read_parquet(f,columns = ['modified_sequence','andromeda_score'])
    df_meta = df_meta[df_meta['andromeda_score']>70]
    df_meta.drop_duplicates('modified_sequence',inplace=True)
    meta_data.append(df_meta)

df_meta = pd.concat(meta_data)

df_meta.drop_duplicates('modified_sequence',inplace=True)


df_meta['sequence'] = df_meta['modified_sequence'].apply(lambda x:internal_without_mods([x])[0])

df_meta['sequence']

df_meta.drop_duplicates('sequence',inplace=True)


In [ ]:
sequence_integer = df_meta["sequence"][...]
n = sequence_integer.shape[0]
n_split = int(n * 0.7)
print("training split at: ", n_split, "/", n)
print("shuffling peptides")
idx = peptide_argsort(sequence_integer, seed=42)
del sequence_integer
print("shuffling train and test")
train_idx, test_idx = argshuffle_splits(idx, n_split)

meta_data_train = df_meta.iloc[train_idx]
meta_data_test = df_meta.iloc[test_idx]

sequence_integer = meta_data_test["sequence"][...]
n = sequence_integer.shape[0]
n_split = int(n * 0.67)
print("training split at: ", n_split, "/", n)
print("shuffling peptides")
idx = peptide_argsort(sequence_integer, seed=42)
del sequence_integer
print("shuffling train and test")
train_idx, test_idx = argshuffle_splits(idx, n_split)

meta_data_val = meta_data_test.iloc[train_idx]
meta_data_ho = meta_data_test.iloc[test_idx]

meta_data_train['sequence'].to_csv('train_sequences.csv')

meta_data_val['sequence'].to_csv('val_sequences.csv')

meta_data_ho['sequence'].to_csv('ho_sequences.csv')

df_meta = df_meta.sample(frac = 1)
